# 01 - Extração dos Dados (SEDAP+)
 ---
## 1. Objetivo

 Este notebook tem como objetivo extrair as tabelas analíticas utilizadas no projeto a partir da plataforma SEDAP+, utilizando consultas SQL sobre os microdados do Censo da Educação Superior.

 ---
## 2. Bases utilizadas
| Base | Ano | Finalidade |
| --- | --- | --- |
| SUP_ALUNO | 2023 | Base principal |
| SUP_CURSO | 2023 | Nome dos cursos |

---

## 3. Consultas SQL

 ---
### 3.1 Distribuição por sexo


In [ ]:
SELECT
    CASE
        WHEN TP_SEXO = 1 THEN 'Masculino'
        WHEN TP_SEXO = 2 THEN 'Feminino'
    END AS SEXO,
    CASE
        WHEN IN_ACAO_AFIRMATIVA = 1
             AND IN_APOIO_SOCIAL = 0
        THEN 'Demanda potencial não atendida'
        ELSE 'Demais estudantes'
    END AS IDPNA,
    COUNT(CO_ALUNO) AS TOTAL
FROM
    `raw.SUP_ALUNO_2024`
WHERE
    CO_IES = 579
GROUP BY
    SEXO,
    IDPNA
ORDER BY
    SEXO,
    IDPNA;

### 3.2 Distribuição por raça

In [ ]:
SELECT
    CASE
        WHEN TP_COR_RACA = 0 THEN 'Não declarado'
        WHEN TP_COR_RACA = 1 THEN 'Branca'
        WHEN TP_COR_RACA = 2 THEN 'Preta'
        WHEN TP_COR_RACA = 3 THEN 'Parda'
        WHEN TP_COR_RACA = 4 THEN 'Amarela'
        WHEN TP_COR_RACA = 5 THEN 'Indígena'
    END AS RACA_COR,
    CASE
        WHEN IN_ACAO_AFIRMATIVA = 1
             AND IN_APOIO_SOCIAL = 0
        THEN 'Demanda potencial não atendida'
        ELSE 'Demais estudantes'
    END AS IDPNA,
    COUNT(CO_ALUNO) AS TOTAL
FROM
    `raw.SUP_ALUNO_2024`
WHERE
    CO_IES = 579
GROUP BY
    RACA_COR,
    IDPNA
ORDER BY
    RACA_COR,
    IDPNA;

### 3.3 IDPNA por curso

In [ ]:
SELECT
    CASE
        WHEN TP_TURNO = 1 THEN 'Matutino'
        WHEN TP_TURNO = 2 THEN 'Vespertino'
        WHEN TP_TURNO = 3 THEN 'Noturno'
        WHEN TP_TURNO = 4 THEN 'Integral'
        ELSE 'Não informado'
    END AS TURNO,

    CASE
        WHEN IN_ACAO_AFIRMATIVA = 1
             AND IN_APOIO_SOCIAL = 0
        THEN 'Demanda potencial não atendida'
        ELSE 'Demais estudantes'
    END AS IDPNA,

    COUNT(CO_ALUNO) AS TOTAL

FROM
    `raw.SUP_ALUNO_2024`

WHERE
    CO_IES = 579

GROUP BY
    TURNO,
    IDPNA

ORDER BY
    TURNO,
    IDPNA;

### 3.4 IDPNA por modalidade

In [ ]:
SELECT

    TP_MODALIDADE_ENSINO,

    CASE
        WHEN IN_ACAO_AFIRMATIVA = 1
             AND IN_APOIO_SOCIAL = 0
        THEN 'Demanda potencial não atendida'
        ELSE 'Demais estudantes'
    END AS IDPNA,

    COUNT(CO_ALUNO) AS TOTAL

FROM
    `raw.SUP_ALUNO_2024`

WHERE
    CO_IES = 579

GROUP BY
    TP_MODALIDADE_ENSINO,
    IDPNA

ORDER BY
    TP_MODALIDADE_ENSINO,
    IDPNA;

### 3.5 IDPNA por escola de origem

In [ ]:
SELECT

    CASE
        WHEN TP_ESCOLA_CONCLUSAO_ENS_MEDIO = 1
        THEN 'Escola pública'
        WHEN TP_ESCOLA_CONCLUSAO_ENS_MEDIO = 2
        THEN 'Escola privada'
        ELSE 'Não informado'
    END AS ESCOLA_ORIGEM,

    CASE
        WHEN IN_ACAO_AFIRMATIVA = 1
             AND IN_APOIO_SOCIAL = 0
        THEN 'Demanda potencial não atendida'
        ELSE 'Demais estudantes'
    END AS IDPNA,

    COUNT(CO_ALUNO) AS TOTAL

FROM
    `raw.SUP_ALUNO_2024`

WHERE
    CO_IES = 579

GROUP BY
    ESCOLA_ORIGEM,
    IDPNA

ORDER BY
    ESCOLA_ORIGEM,
    IDPNA;

### 3.6 IDPNA por ação afirmativa

In [ ]:
SELECT

    TP_ACAO_AFIRMATIVA,

    CASE
        WHEN IN_ACAO_AFIRMATIVA = 1
             AND IN_APOIO_SOCIAL = 0
        THEN 'Demanda potencial não atendida'
        ELSE 'Demais estudantes'
    END AS IDPNA,

    COUNT(CO_ALUNO) AS TOTAL

FROM
    `raw.SUP_ALUNO_2024`

WHERE
    CO_IES = 579

GROUP BY
    TP_ACAO_AFIRMATIVA,
    IDPNA

ORDER BY
    TP_ACAO_AFIRMATIVA,
    IDPNA;

### 3.7 Cursos da UFPB

In [ ]:
SELECT

    CO_CURSO,

    CASE
        WHEN IN_ACAO_AFIRMATIVA = 1
             AND IN_APOIO_SOCIAL = 0
        THEN 'Demanda potencial não atendida'
        ELSE 'Demais estudantes'
    END AS IDPNA,

    COUNT(CO_ALUNO) AS TOTAL

FROM
    `raw.SUP_ALUNO_2024`

WHERE
    CO_IES = 579

GROUP BY
    CO_CURSO,
    IDPNA

ORDER BY
    TOTAL DESC;

## 3.8 - Construção da dimensão de cursos

In [ ]:
SELECT
    a.CO_CURSO,
    c.NO_CURSO,
    c.CO_CINE_ROTULO,
    c.TP_GRAU_ACADEMICO,
    c.TP_MODALIDADE_ENSINO,
    COUNT(a.CO_ALUNO) AS TOTAL_ALUNOS

FROM `raw.SUP_ALUNO_2023` AS a

LEFT JOIN `raw.SUP_CURSO_2023` AS c
    ON CAST(a.CO_CURSO AS STRING) = c.CO_CURSO

WHERE
    a.CO_IES = 579

GROUP BY
    a.CO_CURSO,
    c.NO_CURSO,
    c.CO_CINE_ROTULO,
    c.TP_GRAU_ACADEMICO,
    c.TP_MODALIDADE_ENSINO

ORDER BY
    TOTAL_ALUNOS DESC;

## 3.9 Construção Dimensão Raça

In [ ]:
SELECT
    CO_CURSO,
    TP_COR_RACA,
    COUNT(*) AS TOTAL_ALUNOS
FROM `raw.SUP_ALUNO_2024`
WHERE CO_IES = 579 
GROUP BY CO_CURSO, TP_COR_RACA
ORDER BY CO_CURSO, TP_COR_RACA;

## 4 Construção da Dimensão Sexo

In [ ]:
SELECT
    CO_CURSO,
    TP_SEXO,
    COUNT(*) AS TOTAL_ALUNOS
FROM `raw.SUP_ALUNO_2024`
WHERE CO_IES = 579 
GROUP BY CO_CURSO, TP_SEXO
ORDER BY CO_CURSO, TP_SEXO;

## 4.1 Construção da Dimensão Turno

In [ ]:
SELECT
    CO_CURSO,
    TP_TURNO,
    COUNT(*) AS TOTAL_ALUNOS
FROM `raw.SUP_ALUNO_2024`
WHERE CO_IES = 579 
GROUP BY CO_CURSO, TP_TURNO
ORDER BY CO_CURSO, TP_TURNO;

## 4.2 Construção Fato

In [ ]:
SELECT
    CO_CURSO,
    TP_SEXO,
    TP_COR_RACA,
    TP_TURNO,
    IN_ACAO_AFIRMATIVA,
    IN_APOIO_SOCIAL,
    IN_APOIO_ALIMENTACAO,
    IN_APOIO_MORADIA,
    IN_APOIO_TRANSPORTE,
    IN_APOIO_MATERIAL_DIDATICO,
    IN_APOIO_BOLSA_PERMANENCIA,
    IN_APOIO_BOLSA_TRABALHO,
    COUNT(*) AS TOTAL_ALUNOS
FROM `raw.SUP_ALUNO_2023`
WHERE CO_IES = 579
GROUP BY
    CO_CURSO,
    TP_SEXO,
    TP_COR_RACA,
    TP_TURNO,
    IN_ACAO_AFIRMATIVA,
    IN_APOIO_SOCIAL,
    IN_APOIO_ALIMENTACAO,
    IN_APOIO_MORADIA,
    IN_APOIO_TRANSPORTE,
    IN_APOIO_MATERIAL_DIDATICO,
    IN_APOIO_BOLSA_PERMANENCIA,
    IN_APOIO_BOLSA_TRABALHO
ORDER BY CO_CURSO;

 ---
## 5. Exportação

Depois de executar cada consulta:

Exportar o resultado em CSV.


 ---
## 6. Conclusão
As tabelas exportadas serão utilizadas nas etapas seguintes de integração, tratamento e análise dos dados.